# LLM Request Quickstart

LLM 호출 + MongoDB 캐시를 한 번에 처리하는 `llm_request` 사용 가이드입니다.

| 함수 | 설명 |
| --- | --- |
| `llm_request` | LLM 호출 + 캐시 (sync) |
| `llm_request_async` | LLM 호출 + 캐시 (async) |
| `llm_request_stream` | 스트리밍 LLM 호출 + 캐시 (sync) |
| `llm_request_stream_async` | 스트리밍 LLM 호출 + 캐시 (async) |
| `run_many` | N건 병렬 호출, 완료순 반환 |
| `run_many_stream` | N건 병렬 스트리밍, chunk 실시간 반환 |
| `done_check` | 입력을 해시하여 캐시 키 생성 + DB 조회 |

**사전 조건**
- LLM Handler 서비스 실행 중 (`http://localhost:8100`)
- MongoDB 접근 가능
- `pip install -e ./packages/llm-request` 완료

## 설정

- local로 llm-handler 띄우기


cd /path/to/Kiwoom-Rag

docker compose build llm-handler

docker compose up -d llm-handler

In [1]:
!pip install -e ../llm-handler/sdk
!pip install -e ../mdb-handler/sdk
!pip install -e .

Obtaining file:///home/jovyan/work/Kiwoom-Rag/packages/llm-handler/sdk
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kiwoom-llm-handler (pyproject.toml) ... done
  Created wheel for kiwoom-llm-handler: filename=kiwoom_llm_handler-0.1.0-0.editable-py3-none-any.whl size=3010 sha256=2657cc87753fc0dd14a1311f36531a0f9febef19f7d8cf2bbc6f31b9608eb7de
  Stored in directory: /tmp/pip-ephem-wheel-cache-m5p3pymh/wheels/f6/d5/7c/83a9b72aa1138e62f454f38ae9cc52ab77db00edb5f46b180f
Successfully built kiwoom-llm-handler
  Attempting uninstall: kiwoom-llm-handler
    Found existing installation: kiwoom-llm-handler 0.1.0
    Uninstalling kiwoom-llm-handler-0.1.0:
      Successfully uninstalled kiwoom-llm-handler-0.1.0
Obtaining file:///home/jovyan/work/Kiwoom-Rag/packages/mdb-handler/sdk
  Installing build dependencies .

In [2]:
import os

os.environ["MONGODB_URI"] = "mongodb://192.7.9.45:27017/"
os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
# os.environ["LLM_HANDLER_URL"] = "http://localhost:8100"
os.environ["LLM_HANDLER_URL"] = "http://host.docker.internal:8100"

from mdb_handler import MongoDBHandler
MongoDBHandler.reset()

import llm_handler
llm_handler.configure()

import llm_handler.aio
llm_handler.aio.configure()

print("설정 완료")

설정 완료


> **⚠️ 데이터베이스 안내**
>
> 이 Quickstart에서는 **`kw_ai_dev`** 데이터베이스에 저장됩니다 (개발/테스트용).
> 실제 에이전트 구축 시에는 **`kw_ai`** 로 변경하여 저장해야 합니다.
>
> ```python
> # 개발/테스트
> os.environ["MONGODB_DATABASE"] = "kw_ai_dev"
>
> # 운영/에이전트
> os.environ["MONGODB_DATABASE"] = "kw_ai"
> ```

---
## 1. `llm_request` — 단건 LLM 호출 + 캐시 (sync)

`llm_request`는 `state` dict 하나로 LLM 호출, 캐시 조회, DB 저장을 자동 처리합니다.

### state 구조

```python
state = {
    "llm_input": {          # llm_handler.call()에 전달할 인자
        "input": "질문",
        "model": "gemini-3-flash-preview",
        "system": "시스템 프롬프트",     # 선택
        "temperature": 0,              # 선택
        "max_tokens": 200,             # 선택
        "service_name": "my_service",  # 선택 (메타에 기록)
    },
    "cache_mode": True,     # 캐시 조회 여부 (기본 True)
    "save_mode": True,      # DB 저장 여부 (기본 True)
    "ttl_seconds": 3600,    # 캐시 만료 시간 (선택)
}
```

### 캐시 동작 4가지

| cache_mode | save_mode | 동작 |
| --- | --- | --- |
| `False` | `False` | LLM만 호출, DB 저장 없음 |
| `False` | `True` | LLM 호출 + DB 저장 (캐시 조회 안 함) |
| `True` | `True` | 캐시 조회 → 미스면 LLM 호출 + 저장 |
| `True` | `True` | 캐시 조회 → 히트면 DB에서 바로 반환 |

### 1-1. 기본 호출 (캐시 + 저장 모두 켬)

In [3]:
from llm_request import llm_request

state = {
    "llm_input": {
        "input": "대한민국의 수도는?",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 200,
    },
}

response, meta = llm_request(state)  # sync — await 불필요
print(response.text)
print(f"source: {meta['source']}")  # 'live' 또는 'cache'

대한민국의 수도는 **서울**입니다.
source: live


### 1-2. 동일 입력 재호출 → 캐시 히트

In [4]:
response2, meta2 = llm_request(state)

print(f"source: {meta2['source']}")       # 'cache'
print(f"cache_hit: {meta2['cache_hit']}")  # True
print(f"text: {response2.text}")

source: cache
cache_hit: True
text: 대한민국의 수도는 **서울**입니다.


### 1-3. 캐시 없이 LLM만 호출

In [5]:
state_no_cache = {
    "llm_input": {
        "input": "1+1+1은?",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 200,
    },
    "cache_mode": False,
    "save_mode": True,
}

response3, meta3 = llm_request(state_no_cache)
print(response3.text)
print(f"cache_key: {meta3['cache_key']}")

1+1+1은 **3**입니다.
cache_key: 5531b722c1bd7ebe9280fade70c095816faecf766856e27d278a37975b89dee3


### 1-4. 시스템 프롬프트 + service_name 지정

In [6]:
state_system = {
    "llm_input": {
        "input": "ESG 평가 기준을 알려줘",
        "model": "gemini-3-flash-preview",
        "system": "키움증권 ESG 전문가입니다. 간결하게 답변하세요.",
        "service_name": "esg_assistant",
        "max_tokens": 500,
    },
}

response4, meta4 = llm_request(state_system)
print(response4.text)
print(f"meta: {meta4['meta']}")  # service_name, model, provider 기록됨

키움증권 ESG 전문가입니다. ESG 평가 기준은 크게 세 가지 영역으로 나뉩니다.

**1. 환경(Environment)**
*   **기후변화:** 탄소 배출량, 에너지 효율, 재생에너지 사용.
*   **자원 순환:** 폐기물 관리, 용수 재활용.
*   **환경 경영:** 환경 인증 취득 및 친환경 제품 개발.

**2. 사회(Social)**
*   **인적 자원:** 근로 조건, 안전 보건, 다양성 및 포용성.
*   **공급망:** 협력사 ESG 리스크 관리.
*   **사회 공헌:** 지역사회 기여 및 데이터 보안/개인정보 보호.

**3. 지배구조(Governance)**
*   **이사회:** 구성의 독립성, 전문
meta: {'service_name': 'esg_assistant', 'model': 'gemini-3-flash-preview', 'provider': None}


### 1-5. 반환값 구조

모든 `llm_request` 계열 함수는 `(LLMResponse, meta_dict)` 튜플을 반환합니다.

```python
response, meta = llm_request(state)

# response: LLMResponse — .text, .model, .usage 등
# meta: dict — 아래 필드 포함
{
    "source":     "live" | "cache",
    "cache_hit":  bool,
    "cache_key":  str | None,   # SHA-256 해시 (64자)
    "meta":       dict,         # {service_name, model, provider}
    "collection": str | None,   # 저장된 MongoDB 컬렉션명
}
```

---
## 2. `llm_request_async` — 단건 LLM 호출 + 캐시 (async)

`llm_request`와 동일한 기능의 async 버전입니다. `await`로 호출합니다.
복합 파이프라인(LLM 여러 번 순차 호출)에서 `await`로 이벤트 루프에 양보하여 다른 작업과 병렬 실행할 때 사용합니다.

In [7]:
from llm_request import llm_request_async

response_async, meta_async = await llm_request_async({
    "llm_input": {
        "input": "파이썬의 장점을 한 줄로 알려줘",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 100,
    },
})
print(response_async.text)
print(f"source: {meta_async['source']}")

파이
source: live


---
## 3. `run_many` — N건 병렬 호출, 완료순 반환

N개의 state를 동시에 LLM 호출하고, 먼저 끝나는 것부터 `async for`로 받습니다.

- `concurrency`: 동시 실행 최대 수 (기본 5)
- 각 결과에 `_index` (입력 순서)와 `_state` (원본 state)가 첨부됨

### 3-1. 기본 병렬 호출

In [8]:
from llm_request import run_many

states = [
    {"llm_input": {"input": "대한민국의 수도는?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 100}},
    {"llm_input": {"input": "일본의 수도는?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 100}},
    {"llm_input": {"input": "프랑스의 수도는?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 100}},
]

results = []
async for response, meta in run_many(states, concurrency=3):
    idx = meta["_index"]
    print(f"[완료순] _index={idx}  source={meta['source']}  text={response.text[:50]}")
    results.append((response, meta))

print(f"\n총 {len(results)}건 완료")

[완료순] _index=1  source=live  text=일본의 수도는 **도쿄**입니다.
[완료순] _index=2  source=live  text=프랑스의 수도는 **파리**(Paris)입니다.
[완료순] _index=0  source=live  text=대한민국의 수도는 **서울**입니다.

총 3건 완료


### 3-2. _index로 입력 순서 복원

In [9]:
sorted_results = sorted(results, key=lambda x: x[1]["_index"])

for response, meta in sorted_results:
    print(f"_index={meta['_index']}  input={meta['_state']['llm_input']['input']}")
    print(f"  -> {response.text[:60]}")

_index=0  input=대한민국의 수도는?
  -> 대한민국의 수도는 **서울**입니다.
_index=1  input=일본의 수도는?
  -> 일본의 수도는 **도쿄**입니다.
_index=2  input=프랑스의 수도는?
  -> 프랑스의 수도는 **파리**(Paris)입니다.


### 3-3. 캐시 히트 확인

동일 states를 다시 호출하면 전부 캐시 히트됩니다.

In [10]:
cache_hit_count = 0
async for response, meta in run_many(states, concurrency=3):
    if meta["cache_hit"]:
        cache_hit_count += 1

print(f"캐시 히트: {cache_hit_count}/{len(states)}건")

캐시 히트: 3/3건


### 3-4. _state로 식별자 활용

state에 커스텀 필드(예: `fund_code`)를 넣으면 `_state`에서 꺼내 매핑할 수 있습니다.

In [11]:
states_with_id = [
    {"llm_input": {"input": "삼성전자를 한 줄로 설명해줘", "model": "gemini-3-flash-preview", "max_tokens": 100}, "fund_code": "005930"},
    {"llm_input": {"input": "SK하이닉스를 한 줄로 설명해줘", "model": "gemini-3-flash-preview", "max_tokens": 100}, "fund_code": "000660"},
]

result_map = {}
async for response, meta in run_many(states_with_id, concurrency=2):
    code = meta["_state"]["fund_code"]
    result_map[code] = response.text
    print(f"[{code}] {response.text[:60]}")

print(f"\nresult_map 키: {list(result_map.keys())}")

[000660] **SK
[005930] 대한민국을 대표

result_map 키: ['000660', '005930']


### 3-5. concurrency 비교

`concurrency=1`(직렬)과 `concurrency=3`(병렬)의 속도 차이를 확인합니다.

In [12]:
import time

no_cache_states = [
    {"llm_input": {"input": f"1+{i}은?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 250}, "cache_mode": False, "save_mode": False}
    for i in range(3)
]

t0 = time.time()
async for _, _ in run_many(no_cache_states, concurrency=1):
    pass
serial_time = time.time() - t0

t0 = time.time()
async for _, _ in run_many(no_cache_states, concurrency=3):
    pass
parallel_time = time.time() - t0

print(f"concurrency=1: {serial_time:.2f}s")
print(f"concurrency=3: {parallel_time:.2f}s")
print(f"속도 비: {serial_time / parallel_time:.1f}x")

concurrency=1: 4.81s
concurrency=3: 1.73s
속도 비: 2.8x


### 3-6. 실시간 완료순 반환 확인

응답 시간이 다른 질문 5개를 던져서, 먼저 끝나는 것부터 실시간으로 출력되는지 확인합니다.
짧은 질문과 긴 질문을 섞어서 완료 순서가 입력 순서와 다르게 나오는 것을 눈으로 확인할 수 있습니다.

In [13]:
import time
from llm_request import run_many

realtime_states = [
    {"llm_input": {"input": "1+1은?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 200}, "cache_mode": False, "save_mode": False, "label": "짧은계산"},
    {"llm_input": {"input": "대한민국 경제 성장 역사를 1960년대부터 2020년대까지 연대별로 주요 사건과 함께 상세히 설명해줘. 각 시대의 GDP 성장률, 주요 산업, 경제 위기도 포함해줘.", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 20000}, "cache_mode": True, "save_mode": True, "label": "긴분석1"},
    {"llm_input": {"input": "2+2는?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 200}, "cache_mode": False, "save_mode": False, "label": "짧은계산2"},
    {"llm_input": {"input": "ESG 투자의 역사, 주요 프레임워크(GRI, SASB, TCFD, ISSB), 그린워싱 논란, 그리고 2025년 이후 전망을 각각 500자 이상으로 상세히 설명해줘.", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 20000}, "cache_mode": True, "save_mode": False, "label": "긴분석2"},
    {"llm_input": {"input": "3+3은?", "model": "gemini-3-flash-preview", "temperature": 0, "max_tokens": 200}, "cache_mode": False, "save_mode": False, "label": "짧은계산3"},
]

print("=" * 70)
print(f"총 {len(realtime_states)}건 시작 (concurrency=5)")
print("=" * 70)

t_start = time.time()
async for response, meta in run_many(realtime_states, concurrency=5):
    elapsed = time.time() - t_start
    label = meta["_state"]["label"]
    idx = meta["_index"]
    text_preview = response.text[:40].replace("\n", " ")
    print(f"  [{elapsed:5.2f}s] _index={idx} ({label:6s})  →  {text_preview}...")

total = time.time() - t_start
print("=" * 70)
print(f"전체 완료: {total:.2f}s")
print("짧은 질문이 먼저 나오면 실시간 완료순 반환이 정상 동작하는 것입니다.")

총 5건 시작 (concurrency=5)
  [ 1.78s] _index=2 (짧은계산2 )  →  2 + 2는 **4**입니다....
  [ 1.78s] _index=4 (짧은계산3 )  →  3 + 3은 **6**입니다....
  [ 2.04s] _index=0 (짧은계산  )  →  1+1은 **2**입니다....
  [12.76s] _index=1 (긴분석1  )  →  대한민국의 경제 성장은 전 세계적으로 '한강의 기적'이라 불릴 만큼 유례...
  [15.76s] _index=3 (긴분석2  )  →  ESG(환경·사회·지배구조) 투자에 관한 역사, 프레임워크, 논란 및 전...
전체 완료: 15.76s
짧은 질문이 먼저 나오면 실시간 완료순 반환이 정상 동작하는 것입니다.


---
## 4. `llm_request_stream` — 스트리밍 LLM 호출 (sync)

LLM 응답을 chunk 단위로 실시간 수신합니다. 사용자에게 텍스트가 점진적으로 나타나는 UX에 적합합니다.

항상 `(Iterator[StreamChunk], meta)` 튜플을 반환합니다. 캐시 히트든 미스든 동일한 방식으로 chunk를 순회합니다.

In [14]:
from llm_request import llm_request_stream

state_stream = {
    "llm_input": {
        "input": (
            "다음 주제에 대해 3000자 이상의 상세한 보고서를 작성해줘.\n\n"
            "주제: 인공지능(AI)의 발전 역사와 미래 전망\n\n"
            "1. 1950년대~1970년대 AI 태동기 (튜링 테스트, 퍼셉트론, 첫 번째 AI 겨울)\n"
            "2. 1980년대~1990년대 전문가 시스템과 두 번째 AI 겨울\n"
            "3. 2000년대~2010년대 머신러닝 부활, 딥러닝 혁명 (ImageNet, AlphaGo)\n"
            "4. 2020년대 대규모 언어모델(GPT, Gemini 등)과 생성형 AI\n"
            "5. 향후 10년간 AI가 금융, 의료, 교육 분야에 미칠 영향 예측\n\n"
            "각 시대별로 핵심 인물, 주요 기술, 한계점, 사회적 반응을 포함해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.7,
        "max_tokens": 4000,
    },
    "cache_mode": False,
    "save_mode": True,
}

stream, meta = llm_request_stream(state_stream)
chunk_count = 0
total_text = []
for chunk in stream:
    total_text.append(chunk.text)
    chunk_count += 1
    if chunk_count % 5 == 0:
        print(f"  [chunk #{chunk_count:3d}] ...{chunk.text[:60]!r}", flush=True)
print(f"\n총 {chunk_count}개 chunk 수신, {''.join(total_text).__len__()}자")
print(f"cache_hit: {meta['cache_hit']}, source: {meta['source']}")

  [chunk #  5] ...'닝 혁명, 그리고 미래 10년간 금융, 의료, 교육 분야에 미칠 영향까지 그 발전 역사와 미래 전망을 상세히'
  [chunk # 10] ..."년 다트머스 회의에서 '인공지능(Artificial Intelligence)'이라는 용어를 처음 사용했으며,"
  [chunk # 15] ...'내는 간단한 형태의 모델이었습니다.\n\n*   **주요 기술:**\n    *   **튜링 테스트:** 기계가 '
  [chunk # 20] ...'리 치료사와 대화하는 듯한 착각을 불러일으켰습니다.\n\n*   **한계점:**\n    *   **제한적인 연산'
  [chunk # 25] ...'\n\n*   **사회적 반응:**\n    *   초기에는 AI가 인간의 지능을 모방할 수 있다는 가능성에 대한'
  [chunk # 30] ...' 파이겐바움(Edward Feigenbaum):** 전문가 시스템의 아버지로 불리며, 스탠퍼드 대학에서 MY'
  [chunk # 35] ...' 중 하나입니다.\n    *   **지식 표현(Knowledge Representation):** 전문가 시'
  [chunk # 40] ...' 해결하고 시스템을 유지보수하는 것이 점점 더 어려워졌습니다. 새로운 지식을 추가하는 것도 복잡했습니다.\n '
  [chunk # 45] ...' 고개를 들었습니다.\n    *   결국 1990년대 중반에 이르러 AI 연구 자금 지원이 다시 줄어들고, '

총 47개 chunk 수신, 4753자
cache_hit: False, source: live


### 4-1. 동일 입력 재호출 → 캐시 히트

캐시 히트 시에도 동일하게 `(Iterator[StreamChunk], meta)` 튜플을 반환합니다. 캐시된 텍스트가 단일 StreamChunk로 감싸져 iterator로 제공됩니다.

In [15]:
state_stream_cached = {**state_stream, "cache_mode": True}
stream2, meta2 = llm_request_stream(state_stream_cached)
print(f"cache_hit: {meta2['cache_hit']}")
for chunk in stream2:
    print(chunk.text[:100], "...")

cache_hit: True
## 인공지능(AI)의 발전 역사와 미래 전망: 3000자 이상의 상세 보고서

### 서론

인공지능(Artificial Intelligence, AI)은 인간의 지적 능력을 모 ...


---
## 5. `llm_request_stream_async` — 스트리밍 LLM 호출 (async)

`llm_request_stream`의 async 버전입니다. `await`로 호출하면 `(AsyncIterator[StreamChunk], meta)` 튜플을 반환하고 `async for`로 chunk를 순회합니다.

In [16]:
from llm_request import llm_request_stream_async

state_async_stream = {
    "llm_input": {
        "input": (
            "다음 주제에 대해 3000자 이상의 상세한 기술 분석 보고서를 작성해줘.\n\n"
            "주제: 블록체인 기술의 진화와 산업별 적용 현황\n\n"
            "1. 블록체인 1.0 — 비트코인과 분산 원장의 탄생 (합의 알고리즘, 작업증명)\n"
            "2. 블록체인 2.0 — 이더리움과 스마트 컨트랙트 혁명 (DeFi, NFT)\n"
            "3. 블록체인 3.0 — 확장성 솔루션 (Layer 2, 샤딩, 지분증명 전환)\n"
            "4. 금융 분야: CBDC, 토큰증권(STO), 크로스보더 결제 사례\n"
            "5. 물류·공급망: 원산지 추적, IoT 연계, 위변조 방지\n"
            "6. 의료: 전자건강기록(EHR) 공유, 임상시험 데이터 무결성\n"
            "7. 한계점과 향후 과제 (에너지 문제, 규제, 프라이버시)\n\n"
            "각 섹션마다 구체적인 프로젝트/기업 사례를 2개 이상 포함해줘."
        ),
        "model": "gemini-2.5-flash",
        "temperature": 0.7,
        "max_tokens": 4000,
    },
    "cache_mode": False,
    "save_mode": True,
}

stream, meta = await llm_request_stream_async(state_async_stream)
chunk_count = 0
total_text = []
async for chunk in stream:
    total_text.append(chunk.text)
    chunk_count += 1
    if chunk_count % 5 == 0:
        print(f"  [chunk #{chunk_count:3d}] ...{chunk.text[:60]!r}", flush=True)
print(f"\n총 {chunk_count}개 chunk 수신, {''.join(total_text).__len__()}자")
print(f"cache_hit: {meta['cache_hit']}, source: {meta['source']}")

  [chunk #  5] ...'까지 단계별로 분석하고, 주요 산업 분야에서의 실제 적용 사례와 함께 현재 직면한 한계점 및 향후 과제를 심'
  [chunk # 10] ...' 돕는 메커니즘입니다. 블록체인 1.0의 대표적인 합의 알고리즘은 작업증명(Proof-of-Work, PoW'
  [chunk # 15] ...'된 링크:** 각 블록은 이전 블록의 해시값을 포함하여 암호학적으로 연결됩니다. 따라서 과거 블록의 내용을 '
  [chunk # 20] ...'Litecoin):** 비트코인의 소스 코드를 기반으로 2011년에 개발된 암호화폐입니다. 비트코인과 유사한'
  [chunk # 25] ...'랙트 (Smart Contract):** 미리 정의된 조건이 충족되면 자동으로 실행되는 자체 실행 계약입니다'
  [chunk # 30] ...'. 예술품, 수집품, 게임 아이템 등 디지털 소유권을 증명하는 데 사용됩니다.\n\n**기술적 특징:**\n\n* '
  [chunk # 35] ..." 비트코인이 화폐 기능을 넘어선 '프로그래밍 가능한 돈'이라면, 이더리움은 '프로그래밍 가능한 블록체인'을 "

총 36개 chunk 수신, 3378자
cache_hit: False, source: live


---
## 6. `run_many_stream` — N건 병렬 스트리밍

N개 요청을 동시에 stream으로 실행하고, 모든 요청의 chunk가 실시간으로 섞여서 도착합니다.
각 이벤트에 `_index`가 붙어 어느 요청의 chunk인지 식별할 수 있습니다.

**모든 이벤트에 `meta` dict 포함** — 이벤트 타입에 관계없이 `event["meta"]`로 메타 정보 접근 가능.

**이벤트 타입 4가지:**
- `chunk` — stream 텍스트 조각 (`chunk`, `meta`)
- `done` — 해당 index의 stream 완료 (`meta`)
- `cache_hit` — 캐시 히트, 완성 결과 즉시 전달 (`data`, `meta`)
- `error` — 해당 index 에러 발생 (`error`, `meta`)

In [17]:
from llm_request import run_many_stream

states_stream = [
    {
        "llm_input": {
            "input": (
                "세계 주요 경제 위기의 원인과 결과를 시대순으로 상세히 분석해줘. "
                "1929년 대공황, 1997년 아시아 금융위기, 2008년 글로벌 금융위기, 2020년 코로나 팬데믹 각각에 대해 "
                "발생 배경, 전개 과정, 각국 대응 정책, 경제적·사회적 영향, 그리고 교훈을 서술해줘. 2000자 이상으로."
            ),
            "model": "gemini-2.5-flash", "temperature": 0.7, "max_tokens": 3000,
        },
        "cache_mode": False, "save_mode": True,
    },
    {
        "llm_input": {
            "input": (
                "양자컴퓨팅의 원리와 현재 기술 수준, 그리고 미래 전망을 상세히 설명해줘. "
                "큐비트의 중첩과 얽힘 원리, 양자 게이트, 양자 오류 보정, 주요 기업(IBM·Google·Microsoft)의 로드맵, "
                "암호학·신약개발·최적화 분야의 구체적 활용 가능성, 그리고 양자 우위 달성 현황을 포함해줘. 2000자 이상으로."
            ),
            "model": "gemini-2.5-flash", "temperature": 0.7, "max_tokens": 3000,
        },
        "cache_mode": False, "save_mode": True,
    },
    {
        "llm_input": {
            "input": (
                "한국 반도체 산업의 역사와 글로벌 경쟁 구도를 상세히 분석해줘. "
                "삼성전자와 SK하이닉스의 성장 과정, 메모리 반도체 vs 비메모리 반도체 전략, "
                "TSMC·인텔과의 기술 격차, 미중 반도체 전쟁이 한국에 미치는 영향, "
                "차세대 반도체(HBM, GAA, 2nm 이하 공정)의 기술 동향과 전망을 포함해줘. 2000자 이상으로."
            ),
            "model": "gemini-2.5-flash", "temperature": 0.7, "max_tokens": 3000,
        },
        "cache_mode": False, "save_mode": True,
    },
]

collected = {i: [] for i in range(len(states_stream))}
chunk_counts = {i: 0 for i in range(len(states_stream))}

async for event in run_many_stream(states_stream, concurrency=3):
    idx = event["_index"]
    etype = event["_type"]

    if etype == "chunk":
        collected[idx].append(event["chunk"].text)
        chunk_counts[idx] += 1
        if chunk_counts[idx] % 3 == 0:
            print(f"  [#{idx}] chunk #{chunk_counts[idx]:3d}: {event['chunk'].text[:50]!r}  (source={event['meta']['source']})", flush=True)
    elif etype == "done":
        print(f"  [#{idx}] DONE — {chunk_counts[idx]}개 chunk (cache_hit={event['meta']['cache_hit']})")
    elif etype == "cache_hit":
        response, evt_meta = event["data"]
        print(f"  [#{idx}] CACHE HIT: {response.text[:60]}...  (source={event['meta']['source']})")
    elif etype == "error":
        print(f"  [#{idx}] ERROR: {event['error']}")

print("\n" + "=" * 60)
for idx in sorted(collected):
    full = "".join(collected[idx])
    if full:
        print(f"\n=== 요청 #{idx} (총 {len(full)}자, {chunk_counts[idx]}개 chunk) ===")
        print(full[:200] + "...")

  [#1] chunk #  3: ' 처리하며, 이는 미래 과학기술 및 산업 전반에 걸쳐 막대한 파급 효과를 가져올 것으로 예'  (source=live)
  [#0] chunk #  3: '공황, 1997년 아시아 금융위기, 2008년 글로벌 금융위기, 2020년 코로나 팬데믹에'  (source=live)
  [#1] chunk #  6: '비트는 전자 스핀, 광자의 편광, 초전도 회로의 전류 등 다양한 물리적 현상을 통해 구현될'  (source=live)
  [#0] chunk #  6: ' 열풍이 불어 주가가 기업의 실제 가치를 훨씬 뛰어넘는 수준으로 상승했습니다. 대출을 받아'  (source=live)
  [#1] chunk #  9: '초에 하나의 상태를 계산한다면, 양자 컴퓨터는 $2^N$개의 상태를 동시에 계산하여 병렬 '  (source=live)
  [#1] chunk # 12: '게 하는 핵심 원리 중 하나입니다. 얽힌 큐비트들은 서로 상관관계를 가지며, 이는 복잡한 '  (source=live)
  [#0] chunk #  9: '이 컸으며, 소규모 은행들이 난립하여 전체 시스템이 불안정했습니다.\n*   **금본위제:*'  (source=live)
  [#1] chunk # 15: '하드마르 게이트 (Hadamard Gate, H):** 단일 큐비트를 중첩 상태로 만듭니다'  (source=live)
  [#0] chunk # 12: ' 뉴욕 증시 대폭락을 시작으로 위기가 전 세계로 확산되었습니다.\n*   **주식 시장 붕괴'  (source=live)
  [#2] chunk #  3: ' 한국 반도체 산업의 역사적 흐름, 글로벌 경쟁 구도, 주요 기업의 전략, 미중 반도체 전'  (source=live)
  [#1] chunk # 18: '** 두 개의 큐비트에 작용하며, 제어 큐비트가 1일 때만 대상 큐비트의 상태를 뒤집어 큐'  (source=live)
  [#0] chunk # 15: '들은 수요 감소와 자금 부족으로 생산을

### 6-1. cache hit + miss 혼합

위에서 저장된 캐시가 있으므로, 동일 질문(캐시 히트) + 새 질문(캐시 미스)을 섞어서 보냅니다.
일부는 `cache_hit` 이벤트, 일부는 `chunk` + `done` 이벤트가 됩니다.

In [18]:
states_mixed = [
    # 캐시 히트 예상 (위 섹션 6에서 저장됨)
    {
        "llm_input": {
            "input": (
                "세계 주요 경제 위기의 원인과 결과를 시대순으로 상세히 분석해줘. "
                "1929년 대공황, 1997년 아시아 금융위기, 2008년 글로벌 금융위기, 2020년 코로나 팬데믹 각각에 대해 "
                "발생 배경, 전개 과정, 각국 대응 정책, 경제적·사회적 영향, 그리고 교훈을 서술해줘. 2000자 이상으로."
            ),
            "model": "gemini-2.5-flash", "temperature": 0.7, "max_tokens": 3000,
        },
        "cache_mode": True, "save_mode": True,
    },
    # 캐시 미스 (새 질문)
    {
        "llm_input": {
            "input": (
                "우주의 암흑물질과 암흑에너지가 무엇인지 각각 정의하고, "
                "현재까지 밝혀진 관측 증거와 이론적 배경, 그리고 미해결 질문들을 상세히 설명해줘. 2000자 이상으로."
            ),
            "model": "gemini-2.5-flash", "temperature": 0.7, "max_tokens": 3000,
        },
        "cache_mode": True, "save_mode": True,
    },
    # 캐시 히트 예상 (위 섹션 6에서 저장됨)
    {
        "llm_input": {
            "input": (
                "한국 반도체 산업의 역사와 글로벌 경쟁 구도를 상세히 분석해줘. "
                "삼성전자와 SK하이닉스의 성장 과정, 메모리 반도체 vs 비메모리 반도체 전략, "
                "TSMC·인텔과의 기술 격차, 미중 반도체 전쟁이 한국에 미치는 영향, "
                "차세대 반도체(HBM, GAA, 2nm 이하 공정)의 기술 동향과 전망을 포함해줘. 2000자 이상으로."
            ),
            "model": "gemini-2.5-flash", "temperature": 0.7, "max_tokens": 3000,
        },
        "cache_mode": True, "save_mode": True,
    },
]

print("=== 이벤트 로그 ===")
event_counts = {"chunk": 0, "done": 0, "cache_hit": 0, "error": 0}

async for event in run_many_stream(states_mixed, concurrency=3):
    etype = event["_type"]
    idx = event["_index"]
    event_counts[etype] = event_counts.get(etype, 0) + 1

    if etype == "chunk":
        print(f"  [#{idx}] chunk: {event['chunk'].text[:40]!r}")
    elif etype == "done":
        print(f"  [#{idx}] DONE (stream 완료)")
    elif etype == "cache_hit":
        response, evt_meta = event["data"]
        print(f"  [#{idx}] CACHE HIT: {response.text[:60]}...")
    elif etype == "error":
        print(f"  [#{idx}] ERROR: {event['error']}")

print(f"\n=== 요약 ===")
print(f"  cache_hit: {event_counts['cache_hit']}건 (예상: 2건 — #0, #2)")
print(f"  chunk   : {event_counts['chunk']}건")
print(f"  done    : {event_counts['done']}건 (예상: 1건 — #1)")

=== 이벤트 로그 ===
  [#0] CACHE HIT: 세계 경제는 주기적으로 크고 작은 위기를 겪어왔으며, 각 위기는 고유한 원인과 전개 양상을 보이지만, 동시에...
  [#2] CACHE HIT: 한국 반도체 산업은 지난 수십 년간 눈부신 성장을 거듭하며 세계 경제의 핵심 동력으로 자리매김했습니다. 특히...
  [#1] chunk: '우주의 암흑물질과 암흑에너지는 현대 우주론에서 가장 중요하면서도 미스터리'
  [#1] chunk: ', 우리가 일상에서 접하는 보통 물질(원자, 별, 행성 등)은 단 5%에'
  [#1] chunk: ', 그리고 미해결 질문들을 상세히 설명하겠습니다.\n\n---\n\n## 1. '
  [#1] chunk: '하지 않아 전자기파와 상호작용하지 않으며, 따라서 직접 관측할 수 없는 '
  [#1] chunk: '직 중력을 통해서만 보통 물질과 상호작용하는 것으로 알려져 있으며, 우주'
  [#1] chunk: "는 양성자, 중성자, 전자 등으로 이루어진 '바리온 물질(Baryonic"
  [#1] chunk: ' 증거\n암흑물질의 존재는 다양한 천문학적 관측을 통해 간접적으로 유추되었'
  [#1] chunk: '라 루빈(Vera Rubin) 등의 연구는 나선 은하의 바깥쪽 별들이 뉴'
  [#1] chunk: ' 보이는 물질(별, 가스 등)로만 이루어져 있다면, 중심에서 멀어질수록 '
  [#1] chunk: ' 거의 일정한 속도로 회전하고 있었는데, 이는 은하가 보이지 않는 거대한'
  [#1] chunk: " 프리츠 즈위키(Fritz Zwicky)가 처음 '잃어버린 질량(miss"
  [#1] chunk: ' 은하들의 무작위적인 움직임(속도 분산)을 분석하면, 은하단을 중력적으로'
  [#1] chunk: ' 있습니다. 이는 은하단에도 암흑물질이 풍부하다는 증거입니다.\n\n*   '
  [#1] chunk: ' 거대한 질량은 시공간을 휘게 하여 그 뒤에 있는 빛을 휘게 만듭니다. '
  [#1] chunk: ' 

---
## 7. `done_check` — 캐시 키 생성 + DB 조회

`done_check`는 `llm_request` 내부에서도 쓰이지만, 단독으로도 활용할 수 있습니다.

- 동일 입력이 이미 처리됐는지 확인할 때
- 캐시 키만 미리 계산할 때
- 저장된 결과를 직접 조회할 때

### 7-1. 캐시 키만 생성 (DB 조회 없이)

In [19]:
from llm_request import done_check

state = {
    "llm_input": {
        "input": "대한민국의 수도는?",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 200,
    },
}

dc = done_check(state, lookup_cache=False)

print(f"cache_key:  {dc.cache_key}")
print(f"collection: {dc.collection}")
print(f"cached:     {dc.cached}")  # None (조회 안 했으므로)

cache_key:  42768f11620c3c49f6d1708a2cdcdd2f90fc48359bd62debd2fe115e2d636818
collection: agent_llm
cached:     None


### 7-2. 캐시 조회까지 수행

In [20]:
dc2 = done_check(state, lookup_cache=True)

if dc2.cached:
    print("캐시 히트!")
    print(f"  text: {dc2.cached['result']['text'][:80]}...")
else:
    print("캐시 미스 — 아직 저장된 결과 없음")

print(f"cache_error: {dc2.cache_error}")  # DB 연결 오류 시 True

캐시 히트!
  text: 대한민국의 수도는 **서울**입니다....
cache_error: False


### 7-3. 캐시 키 동작 원리

`llm_input`에서 캐시에 영향을 주는 필드만 추출 → 정규화 → SHA-256 해시로 캐시 키를 생성합니다.

**캐시 키에 포함되는 필드:**
`input`, `system`, `history`, `model`, `max_tokens`, `temperature`, `cached_content`, `service_name`, `seed`, `output`, `thinking`, `tools`

이 중 하나라도 다르면 **다른 캐시 키**가 됩니다.

In [21]:
dc_a = done_check({"llm_input": {"input": "hello", "model": "gemini-3-flash-preview"}}, lookup_cache=False)
dc_b = done_check({"llm_input": {"input": "hello", "model": "gemini-3-flash-preview"}}, lookup_cache=False)
print(f"같은 입력: {dc_a.cache_key == dc_b.cache_key}")  # True

dc_c = done_check({"llm_input": {"input": "hello", "model": "gemini-2.0-flash"}}, lookup_cache=False)
print(f"다른 모델: {dc_a.cache_key == dc_c.cache_key}")  # False

같은 입력: True
다른 모델: False


### 7-4. 반환값 구조 — `DoneCheckResult`

```python
@dataclass(frozen=True)
class DoneCheckResult:
    collection: str              # 저장 대상 MongoDB 컬렉션명
    cache_key: str               # SHA-256 해시 (64자)
    canonical_json: str          # 정규화된 입력 JSON 문자열
    cached: dict | None          # 캐시 히트 시 저장된 문서, 미스 시 None
    cache_error: bool            # DB 조회 에러 발생 여부
```

---
# Appendix: RM 위험등급 산정 시뮬레이션

RM 위험등급 산정과 유사한 구조를 `llm_request_async`로 구현한 시뮬레이션입니다.

**패턴 C (복합 파이프라인)** 의 실제 적용 예시로, 각 펀드 내부에서 LLM을 여러 번 순차 호출하되
펀드 간에는 `asyncio.Semaphore` + `asyncio.as_completed`로 동시 실행합니다.

```
펀드1: [DB 조회] → [await LLM #1: 분류] → [중간 처리] → [await LLM #2: 등급] → 결과
펀드2:            [DB 조회] → [await LLM #1: 분류] → [중간 처리] → [await LLM #2: 등급] → 결과
펀드3:                       [DB 조회] → [await LLM #1: 분류] → ...                    → 결과
펀드4:                                  [DB 조회] → [await LLM #1: 분류] → ...         → 결과
```

`await` 지점에서 이벤트 루프에 양보하므로, 한 펀드가 LLM 응답을 기다리는 동안 다른 펀드가 진행됩니다.

> `run_many`는 state 목록을 받아 펀드당 LLM 1회 호출하는 단순 병렬에 적합합니다.
> 반면 이 패턴은 **LLM 여러 번 순차 호출 + 이전 결과에 다음이 의존**하는 복합 파이프라인에 사용합니다.

핵심: Step2는 Step1 결과에 의존하므로 `run_many`로 묶을 수 없고, 펀드 단위로 순차 호출해야 합니다.
대신 **펀드 간**은 `await` 양보 덕분에 자연스럽게 병렬 실행됩니다.

## A-1. 가상 펀드 데이터

실제로는 `winus_handler`(Oracle)에서 조회하지만, 여기서는 가상 데이터로 대체합니다.

In [22]:
FUNDS = [
    {"code": "005930", "name": "삼성전자", "type": "주식형", "risk_factor": "반도체 사이클"},
    {"code": "000660", "name": "SK하이닉스", "type": "주식형", "risk_factor": "메모리 가격 변동"},
    {"code": "035420", "name": "NAVER", "type": "혼합형", "risk_factor": "광고 매출 의존"},
    {"code": "068270", "name": "셀트리온", "type": "주식형", "risk_factor": "바이오시밀러 경쟁"},
]

print(f"평가 대상: {len(FUNDS)}개 펀드")
for f in FUNDS:
    print(f"  [{f['code']}] {f['name']} ({f['type']}, 리스크: {f['risk_factor']})")

평가 대상: 4개 펀드
  [005930] 삼성전자 (주식형, 리스크: 반도체 사이클)
  [000660] SK하이닉스 (주식형, 리스크: 메모리 가격 변동)
  [035420] NAVER (혼합형, 리스크: 광고 매출 의존)
  [068270] 셀트리온 (주식형, 리스크: 바이오시밀러 경쟁)


## A-2. 펀드별 평가 함수 (async)

각 펀드에 대해 LLM을 2번 순차 호출하되, `await`로 이벤트 루프에 양보합니다.

```
evaluate_fund()
  ├─ [sync] DB 조회 (여기서는 가상 데이터)
  ├─ [await] LLM #1: 운용유형 분류
  ├─ [sync] 중간 처리 (다음 프롬프트 구성)
  ├─ [await] LLM #2: 위험등급 산정 (Step1 결과에 의존)
  └─ [sync] 등급 산출
```

In [23]:
import time
from llm_request import llm_request_async


async def evaluate_fund(fund: dict) -> dict:
    """펀드 1개에 대해 LLM 2번 순차 호출 → 등급 산정."""
    code, name = fund["code"], fund["name"]
    t0 = time.time()

    # ── Step 1: 운용유형 분류 (LLM #1) ──
    r1, m1 = await llm_request_async({
        "llm_input": {
            "input": (
                f"다음 펀드의 투자 유형을 "
                f"'공격투자형/적극투자형/위험중립형/안정추구형/안정형' 중 하나로 분류하고 "
                f"이유를 1문장으로 설명해줘.\n"
                f"펀드명: {name}\n유형: {fund['type']}\n주요 리스크: {fund['risk_factor']}"
            ),
            "model": "gemini-3-flash-preview",
            "temperature": 0,
            "max_tokens": 200,
        },
        "cache_mode": False,
        "save_mode": False,
    })
    step1_time = time.time() - t0
    classification = r1.text
    print(f"    [{code} {name}] Step1 분류 완료 ({step1_time:.1f}s)")

    # ── Step 2: 위험등급 산정 (LLM #2, Step1 결과에 의존) ──
    r2, m2 = await llm_request_async({
        "llm_input": {
            "input": (
                f"다음 펀드의 위험등급을 G(양호)/Y(주의)/A(경고)/R(위험) 중 하나로 산정하고 "
                f"근거를 3줄로 설명해줘.\n"
                f"펀드명: {name}\n투자유형 분류: {classification[:100]}\n"
                f"주요 리스크: {fund['risk_factor']}"
            ),
            "model": "gemini-3-flash-preview",
            "temperature": 0,
            "max_tokens": 300,
        },
        "cache_mode": False,
        "save_mode": False,
    })
    total_time = time.time() - t0
    grade_text = r2.text
    print(f"    [{code} {name}] Step2 등급 완료 ({total_time:.1f}s)")

    return {
        "code": code,
        "name": name,
        "classification": classification,
        "grade": grade_text,
        "elapsed": total_time,
    }

print("evaluate_fund 정의 완료")

evaluate_fund 정의 완료


## A-3. 병렬 실행 (Semaphore + as_completed)

4개 펀드를 동시에 실행합니다. 각 펀드 내부의 `await llm_request_async()`에서 이벤트 루프에 양보하므로,
한 펀드가 LLM 응답을 기다리는 동안 다른 펀드의 코드가 실행됩니다.

실행 결과에서 Step1/Step2 완료 메시지가 **서로 다른 펀드끼리 인터리빙**되는 것을 확인하세요.

In [24]:
import asyncio

CONCURRENCY = 4

sem = asyncio.Semaphore(CONCURRENCY)

async def limited(fund):
    async with sem:
        return await evaluate_fund(fund)

print("=" * 70)
print(f"{len(FUNDS)}개 펀드 등급 산정 시작 (concurrency={CONCURRENCY})")
print(f"각 펀드: LLM 2회 순차 호출, 펀드 간 병렬")
print("=" * 70)

t_start = time.time()
tasks = [asyncio.create_task(limited(f)) for f in FUNDS]

pipeline_results = []
for coro in asyncio.as_completed(tasks):
    result = await coro
    elapsed = time.time() - t_start
    print(f"  >> [{elapsed:5.2f}s] {result['name']} 전체 완료")
    pipeline_results.append(result)

total = time.time() - t_start
print("=" * 70)
print(f"전체 완료: {total:.2f}s")

4개 펀드 등급 산정 시작 (concurrency=4)
각 펀드: LLM 2회 순차 호출, 펀드 간 병렬
    [068270 셀트리온] Step1 분류 완료 (2.0s)
    [035420 NAVER] Step1 분류 완료 (2.2s)
    [000660 SK하이닉스] Step1 분류 완료 (2.2s)
    [005930 삼성전자] Step1 분류 완료 (2.6s)
    [068270 셀트리온] Step2 등급 완료 (4.6s)
  >> [ 4.62s] 셀트리온 전체 완료
    [000660 SK하이닉스] Step2 등급 완료 (4.9s)
  >> [ 4.90s] SK하이닉스 전체 완료
    [035420 NAVER] Step2 등급 완료 (4.9s)
  >> [ 4.91s] NAVER 전체 완료
    [005930 삼성전자] Step2 등급 완료 (5.0s)
  >> [ 5.00s] 삼성전자 전체 완료
전체 완료: 5.00s


## A-4. 결과 요약

코드 순서대로 정렬하여 각 펀드의 분류 결과와 위험등급을 표시합니다.

In [25]:
for r in sorted(pipeline_results, key=lambda x: x["code"]):
    print(f"[{r['code']} {r['name']}] ({r['elapsed']:.1f}s)")
    print(f"  분류: {r['classification'][:80]}")
    print(f"  등급: {r['grade'][:80]}")
    print()

[000660 SK하이닉스] (4.9s)
  분류: 분류: **공격
  등급: 해당 펀드(SK하이닉스 집중

[005930 삼성전자] (5.0s)
  분류: 분류: **공격투자
  등급: 해당 펀드(삼성전자)의 위험

[035420 NAVER] (4.9s)
  분류:  Advertising dependency:* This indicates business risk associated
  등급: 해당 펀드(NAVER)의

[068270 셀트리온] (4.6s)
  분류: 투자 유형: **공격투자
  등급: 해당 펀드(투자 대상)의



## A-5. 참고: 실제 RM 적용 시 변경점

이 시뮬레이션을 실제 RM에 적용할 때:

1. **DB 조회**: `FUNDS` 리스트 대신 `winus_handler.query_to_dataframe()`로 Oracle에서 펀드 정보 조회 (sync, 수 ms)
2. **프롬프트**: 실제 체크리스트, 리스크 데이터, 과거 이력 등을 프롬프트에 포함
3. **LLM 호출 횟수**: 시뮬레이션은 2회지만, 실제로는 분류 → 정성 → 검증 등 더 많을 수 있음
4. **결과 저장**: `save_mode=True`로 캐시 저장하여 동일 입력 재호출 방지
5. **SSE 전송**: `as_completed` 루프 안에서 SSE 이벤트를 yield하여 프론트엔드에 실시간 진행률 전달

**핵심은 동일**: `async def evaluate_fund()` 안에서 `await llm_request_async()`를 호출하고,
여러 펀드를 `Semaphore` + `as_completed`로 동시 실행하는 구조.

---
## 정리

테스트 데이터를 삭제합니다.

In [27]:
mdb = MongoDBHandler()
col = mdb.get_database("kw_ai_dev", "agent_llm")

deleted = 0

# 첫 번째 llm_request 결과 (max_tokens=200)
first_state = {
    "llm_input": {
        "input": "대한민국의 수도는?",
        "model": "gemini-3-flash-preview",
        "temperature": 0,
        "max_tokens": 200,
    },
}
dc = done_check(first_state, lookup_cache=False)
res = col.delete_one({"_id": dc.cache_key})
deleted += res

# run_many 결과
for response, meta in results:
    if meta.get("cache_key"):
        res = col.delete_one({"_id": meta["cache_key"]})
        deleted += res

# 단건 호출 결과
for m in [meta, meta3, meta4, meta_async]:
    if m.get("cache_key"):
        res = col.delete_one({"_id": m["cache_key"]})
        deleted += res

# stream 결과 (done_check로 캐시 키 추출하여 삭제)
for st in [state_stream, state_async_stream] + states_stream:
    dc = done_check(st, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res

# 실시간 완료순 / 혼합 스트림 결과
for st in [realtime_states[1], states_mixed[1]]:
    dc = done_check(st, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res

# states_with_id 결과 (Cell 26)
for st in states_with_id:
    dc = done_check(st, lookup_cache=False)
    res = col.delete_one({"_id": dc.cache_key})
    deleted += res


print(f"정리 완료: {deleted}건 삭제")

정리 완료: 1건 삭제
